# Data Sources

In [1]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
BASE = Path.cwd()
if not (BASE / "evidence").exists():
    BASE = BASE / "notebooks" 
assert (BASE / "evidence").is_dir(), "Open from the project root or notebooks."
EVIDENCE = BASE / "evidence"
def read_json(name):
    return json.loads((EVIDENCE / name).read_text(encoding="utf-8-sig"))
pd.set_option("display.max_colwidth", 90)
plt.rcParams.update({"figure.figsize": (10, 4.5), "axes.spines.top": False,
                     "axes.spines.right": False, "font.size": 11})


## Source records
The source tabs have different grains. Joining their rows and summing everything would duplicate sales or visits. Recorded sales are package prices, not profit or settled cash. The PAYMENTS tab is not sufficiently maintained to act as the payment-tracking authority.

In [2]:
sources = pd.DataFrame([
    ["Sales", "One sale/package record", "Package, price, customer and service linkage"],
    ["Commercial Clients", "One source client/contract record", "Commercial context and contract matching"],
    ["Calendar", "One event in a particular source snapshot", "Bookings, cancellations, service stages and recorded returns"],
    ["Recurring Payments", "One tracked contract, with reminder-date columns", "Reminder dates and recorded cumulative paid/balance amounts"],
    ["Payments", "One recorded payment entry", "Coverage/reconciliation checks; not reliable current paid status"],
    ["Refund", "One source refund record", "Refunds shown separately from sales"],
    ["Weather and geography", "Location/date or mapped feature", "Environmental predictors with availability limits"]
], columns=["Source", "One row means", "Use"])
display(sources)
columns = pd.read_csv(EVIDENCE / "source_column_dictionary.csv")
display(columns.groupby("source_tab", as_index=False).size().rename(columns={"size":"Documented columns"}))

Source,One row means,Use
Sales,One sale/package record,"Package, price, customer and service linkage"
Commercial Clients,One source client/contract record,Commercial context and contract matching
Calendar,One event in a particular source snapshot,"Bookings, cancellations, service stages and recorded returns"
Recurring Payments,"One tracked contract, with reminder-date columns",Reminder dates and recorded cumulative paid/balance amounts
Payments,One recorded payment entry,Coverage/reconciliation checks; not reliable current paid status
Refund,One source refund record,Refunds shown separately from sales
Weather and geography,Location/date or mapped feature,Environmental predictors with availability limits


source_tab,Documented columns
B2B FOLLOW UP,25
PAYMENTS,12
SALES,43


## External data
The following register records identified sources and their limits. Identified or downloaded does not automatically mean retained in the frozen model. Actual building age is unavailable; neighbourhood settlement age is a different proxy. Online silence does not mean a building has no pest history.

In [3]:
external = pd.DataFrame(read_json("source_catalog.json"))
display(external[["predictor", "source", "status", "time_limit"]])

predictor,source,status,time_limit
Nearby food outlets,Geoapify Places / OpenStreetMap,Live collection against archived environmental locations; summary saved separately,"Current map snapshot, opening/closing history not established"
Building construction year,OpenStreetMap building date tags via Geoapify,100-location date-tag coverage check; summary saved separately,Business opening date must not be mistaken for construction date
Past pest reports,MOH / MYSA iDengue,Official mosquito/dengue context source identified; no general building pest-history dataset verified,"Dengue cases are not cockroach, rodent or general infestation labels"
Sewage treatment plants,IWK official facilities/contact addresses,Named facility addresses found; office/payment-counter addresses must be excluded,Inventory completeness and facility operating dates unverified
Sewage assets,IWK Indah Geoportal,"Official asset source, paid access advertised; no purchase made","License, data dates and access require review"
Drinking-water treatment plants,Air Selangor Sustainability Report 2022,Official historical named inventory and regional map found (34 named plants),Not a ready customer-linked coordinate dataset; sewage and drinking-water plants differ
Restaurant locations,MyGeoMap official REST layer,Metadata discovered; direct web retrieval timed out,Restaurant existence dates and reuse conditions not verified
"Food, facilities and building dates",Overpass OSM API,Direct bounded POST failed 502; mirror 500; GET probes returned 406/429,No usable download; no false zero coverage recorded


## Warehouse layers
`bronze`: immutable source snapshots. `silver`: cleaned and linked records. `gold`: approved management views. `analytics_ml`: historical research tables. `audit`: pipeline and collection evidence. `quality`: data checks.

Source extraction: `scripts/load_operational_bronze.py`, `scripts/load_calendar_bronze.py`. Live reporting entry point: `scripts/cp2_reporting_release.py`. See `planning/BIGQUERY_START_HERE.md` and the live warehouse catalogue in `project_guide`.

Credentials are not needed to read this notebook. Live extraction requires the project's approved read-only Google source access.